# ✦ LumenTrail: follow the evidence

This short lab uses **synthetic** health facts. The offline hash vectors test the retrieval pipeline, but do not understand meaning. Run from the project root.

In [ ]:
import sys
import tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))
from lumentrail.datasets import DEMO_JUDGEMENTS, DEMO_PASSAGES
from lumentrail.embedding import HashEmbedder
from lumentrail.evaluation import evaluate
from lumentrail.retrieval import search
from lumentrail.storage import connect, replace_passages
folder = tempfile.TemporaryDirectory()
database = connect(Path(folder.name) / 'lab.db')
embedder = HashEmbedder()
texts = [f"{item['question']} {item['answer']}" for item in DEMO_PASSAGES]
replace_passages(database, DEMO_PASSAGES, embedder.encode(texts), embedder.name)

## Compare retrieval routes

The exact term `AF` should favour keyword search. Hybrid uses reciprocal rank fusion to combine keyword and vector ranks. Inspect the IDs, not just the scores.

In [ ]:
for method in ('keyword', 'dense', 'hybrid'):
    hits = search(database, 'What is AF?', embedder, method=method, limit=2)
    print(method, [hit['id'] for hit in hits])

## A useful failure

Ask for an irregular heartbeat without saying `AF`. The hash baseline cannot learn that these phrases mean the same thing. Inspect the hits before trusting a score, then try a real embedding with `lumentrail ingest ... --embedding bge`.

In [ ]:
[(hit['id'], hit['score']) for hit in search(database, 'irregular heartbeat', embedder, method='hybrid', limit=3)]

## Measure, then inspect misses

These three labels make a smoke test, not a clinical benchmark. HealthSearchQA provides questions without labels, so review relevant passage IDs before evaluating it.

In [ ]:
for method in ('keyword', 'dense', 'hybrid'):
    report = evaluate(database, DEMO_JUDGEMENTS, embedder, method=method, limit=2)
    print(method, report['recall_at_k'], report['mean_reciprocal_rank'])
database.close()
folder.cleanup()